In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import json
import pandas as pd
from PIL import Image
import clip
import torch
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace
from tqdm import tqdm
from train import CLIP_Clean_Train
from dci import JsonDCIDataset
from docci import DocciDataset
# from sharegpt4v_test import JsonSHAREGPT4VTESTDataset

# 1) Thiết lập multiprocessing start method
# import torch.multiprocessing as mp
# try:
#     mp.set_start_method('spawn', force=True)
# except RuntimeError:
#     pass

class Flickr30kPairDataset(Dataset):
    """
    Mỗi caption thành một sample riêng, và chỉ lấy split bạn muốn (train/val/test).
    Trả về: image_tensor, caption_long, caption_short, img_path, split
    """
    def __init__(
        self,
        csv_path: str,
        root_dir: str,
        clip_model: str = "ViT-B/16",
        split: str = None,
        max_items: int = None,
    ):
        super().__init__()
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        if max_items is not None:
            df = df.iloc[:max_items]
        df = df.reset_index(drop=True)

        samples = []
        for _, row in df.iterrows():
            img_path = os.path.join(root_dir, "flickr30k-images", row["filename"])
            caps = json.loads(row["raw"])
            # for cap in caps:
            samples.append({
                "img_path": img_path,
                "caption": caps[0].strip(),
                "split": row["split"],
                "img_id": row["img_id"],
            })
        self.samples = samples

        _, self.preprocess = clip.load(clip_model, device="cpu")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["img_path"]).convert("RGB")
        image_tensor = self.preprocess(image)
        caption_long = item["caption"]
        caption_short = item["caption"]
        return image_tensor, caption_long, caption_short, item["img_path"], item["split"]
    


# csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
# root = "/home/ubuntu/shared/hieu.tq/flickr30k"
# ds = Flickr30kPairDataset(csv_file, root, max_items=100)
# loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=True)

# batch = next(iter(loader))

# print("Batch images:", batch["image"].shape)    # [16,3,224,224]
# print("Batch captions:", batch["caption"][:5]) # 5 câu đầu
# print("Paths:", batch["img_path"][:3])

if __name__ == "__main__":
    # --- Inline args và load model ---
    args = SimpleNamespace(
        base_model='ViT-B/16',
        download_root=None,
        log_scale=4.6052,
        batch_size=64,
        epochs=1,
        lr=1e-5,
        refine_lr_image=2e-6,
        refine_lr_text=2e-4,
        weight_decay=0.01,
        warmup_length=200,
        exp_name='eval_run',
        ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--16_36_34_-docci-spm-no-divide.pt"
    )

    trainer = CLIP_Clean_Train(args)
    state = torch.load(args.ckpt_path, map_location='cpu')
    trainer.model.load_state_dict(state, strict=False)

    # --- Khởi tạo dataset + loader ---
    csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
    root_dir = "/home/ubuntu/shared/hieu.tq/flickr30k"
    test_ds = Flickr30kPairDataset(
        csv_path=csv_file,
        root_dir=root_dir,
        clip_model="ViT-B/16",
        split="test",
        max_items=None
    )
    # test_ds = DocciDataset(split='test', max_items=None)

    test_loader = DataLoader(
        test_ds,
        batch_size   = args.batch_size,
        shuffle      = False,
        num_workers  = 32,     # bây giờ spawn sẽ cho phép dùng CUDA safe
        pin_memory   = True,
    )

    print(f"Số mẫu trong test set: {len(test_loader.dataset)}")
    print(f"Số batch trong test loader: {len(test_loader)}")

    # --- Chạy đánh giá ---
    metrics = trainer.test_epoch_ver4(test_loader, epoch=0)
    print(f"Individual metrics: {metrics}")


Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:25<00:00,  1.58s/it]

▶ Full Text→Image:     76.5000%
▶ Image→Full Text:     79.5000%
▶ 1th Sentence→Image:  76.3000%
▶ 2th Sentence→Image:  0.2000%
▶ 3th Sentence→Image:  0.1000%
▶ 4th Sentence→Image:  0.1000%
Individual metrics: [0.765, 0.795, 0.763, 0.002, 0.001, 0.001]


In [4]:
import os
import json
import pandas as pd
from PIL import Image
import clip
import torch
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace
from tqdm import tqdm
from train import CLIP_Clean_Train
from dci import JsonDCIDataset
from docci import DocciDataset
# from sharegpt4v_test import JsonSHAREGPT4VTESTDataset

# 1) Thiết lập multiprocessing start method
# import torch.multiprocessing as mp
# try:
#     mp.set_start_method('spawn', force=True)
# except RuntimeError:
#     pass

class Flickr30kPairDataset(Dataset):
    """
    Mỗi caption thành một sample riêng, và chỉ lấy split bạn muốn (train/val/test).
    Trả về: image_tensor, caption_long, caption_short, img_path, split
    """
    def __init__(
        self,
        csv_path: str,
        root_dir: str,
        clip_model: str = "ViT-B/16",
        split: str = None,
        max_items: int = None,
    ):
        super().__init__()
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        if max_items is not None:
            df = df.iloc[:max_items]
        df = df.reset_index(drop=True)

        samples = []
        for _, row in df.iterrows():
            img_path = os.path.join(root_dir, "flickr30k-images", row["filename"])
            caps = json.loads(row["raw"])
            # for cap in caps:
            samples.append({
                "img_path": img_path,
                "caption": caps[0].strip(),
                "split": row["split"],
                "img_id": row["img_id"],
            })
        self.samples = samples

        _, self.preprocess = clip.load(clip_model, device="cpu")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["img_path"]).convert("RGB")
        image_tensor = self.preprocess(image)
        caption_long = item["caption"]
        caption_short = item["caption"]
        return image_tensor, caption_long, caption_short, item["img_path"], item["split"]
    


# csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
# root = "/home/ubuntu/shared/hieu.tq/flickr30k"
# ds = Flickr30kPairDataset(csv_file, root, max_items=100)
# loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=True)

# batch = next(iter(loader))

# print("Batch images:", batch["image"].shape)    # [16,3,224,224]
# print("Batch captions:", batch["caption"][:5]) # 5 câu đầu
# print("Paths:", batch["img_path"][:3])

if __name__ == "__main__":
    # --- Inline args và load model ---
    args = SimpleNamespace(
        base_model='ViT-B/16',
        download_root=None,
        log_scale=4.6052,
        batch_size=64,
        epochs=1,
        lr=1e-5,
        refine_lr_image=2e-6,
        refine_lr_text=2e-4,
        weight_decay=0.01,
        warmup_length=200,
        exp_name='eval_run',
        ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-21--14_20_01_-dci-spm-no-divide-5.pt"
    )

    trainer = CLIP_Clean_Train(args)
    state = torch.load(args.ckpt_path, map_location='cpu')
    trainer.model.load_state_dict(state, strict=False)

    # --- Khởi tạo dataset + loader ---
    csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
    root_dir = "/home/ubuntu/shared/hieu.tq/flickr30k"
    test_ds = Flickr30kPairDataset(
        csv_path=csv_file,
        root_dir=root_dir,
        clip_model="ViT-B/16",
        split="test",
        max_items=None
    )
    # test_ds = DocciDataset(split='test', max_items=None)

    test_loader = DataLoader(
        test_ds,
        batch_size   = args.batch_size,
        shuffle      = False,
        num_workers  = 32,     # bây giờ spawn sẽ cho phép dùng CUDA safe
        pin_memory   = True,
    )

    print(f"Số mẫu trong test set: {len(test_loader.dataset)}")
    print(f"Số batch trong test loader: {len(test_loader)}")

    # --- Chạy đánh giá ---
    metrics = trainer.test_epoch_ver4(test_loader, epoch=0)
    print(f"Individual metrics: {metrics}")


Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:17<00:00,  1.09s/it]

▶ Full Text→Image:     73.4000%
▶ Image→Full Text:     76.4000%
▶ 1th Sentence→Image:  73.6000%
▶ 2th Sentence→Image:  0.2000%
▶ 3th Sentence→Image:  0.1000%
▶ 4th Sentence→Image:  0.1000%
Individual metrics: [0.734, 0.764, 0.736, 0.002, 0.001, 0.001]


In [ ]:
import os
import json
import pandas as pd
from PIL import Image
import clip
import torch
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace
from tqdm import tqdm
from train import CLIP_Clean_Train
from dci import JsonDCIDataset
from docci import DocciDataset
# from sharegpt4v_test import JsonSHAREGPT4VTESTDataset

# 1) Thiết lập multiprocessing start method
# import torch.multiprocessing as mp
# try:
#     mp.set_start_method('spawn', force=True)
# except RuntimeError:
#     pass

class Flickr30kPairDataset(Dataset):
    """
    Mỗi caption thành một sample riêng, và chỉ lấy split bạn muốn (train/val/test).
    Trả về: image_tensor, caption_long, caption_short, img_path, split
    """
    def __init__(
        self,
        csv_path: str,
        root_dir: str,
        clip_model: str = "ViT-B/16",
        split: str = None,
        max_items: int = None,
    ):
        super().__init__()
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        if max_items is not None:
            df = df.iloc[:max_items]
        df = df.reset_index(drop=True)

        samples = []
        for _, row in df.iterrows():
            img_path = os.path.join(root_dir, "flickr30k-images", row["filename"])
            caps = json.loads(row["raw"])
            # for cap in caps:
            samples.append({
                "img_path": img_path,
                "caption": caps[0].strip(),
                "split": row["split"],
                "img_id": row["img_id"],
            })
        self.samples = samples

        _, self.preprocess = clip.load(clip_model, device="cpu")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["img_path"]).convert("RGB")
        image_tensor = self.preprocess(image)
        caption_long = item["caption"]
        caption_short = item["caption"]
        return image_tensor, caption_long, caption_short, item["img_path"], item["split"]
    


# csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
# root = "/home/ubuntu/shared/hieu.tq/flickr30k"
# ds = Flickr30kPairDataset(csv_file, root, max_items=100)
# loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=True)

# batch = next(iter(loader))

# print("Batch images:", batch["image"].shape)    # [16,3,224,224]
# print("Batch captions:", batch["caption"][:5]) # 5 câu đầu
# print("Paths:", batch["img_path"][:3])

if __name__ == "__main__":
    # --- Inline args và load model ---
    args = SimpleNamespace(
        base_model='ViT-B/16',
        download_root=None,
        log_scale=4.6052,
        batch_size=64,
        epochs=1,
        lr=1e-5,
        refine_lr_image=2e-6,
        refine_lr_text=2e-4,
        weight_decay=0.01,
        warmup_length=200,
        exp_name='eval_run',
        ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-22--05_40_36_-dci-spm-no-divide-5.pt"
    )

    trainer = CLIP_Clean_Train(args)
    state = torch.load(args.ckpt_path, map_location='cpu')
    trainer.model.load_state_dict(state, strict=False)

    # --- Khởi tạo dataset + loader ---
    csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
    root_dir = "/home/ubuntu/shared/hieu.tq/flickr30k"
    test_ds = Flickr30kPairDataset(
        csv_path=csv_file,
        root_dir=root_dir,
        clip_model="ViT-B/16",
        split="test",
        max_items=None
    )
    # test_ds = DocciDataset(split='test', max_items=None)

    test_loader = DataLoader(
        test_ds,
        batch_size   = args.batch_size,
        shuffle      = False,
        num_workers  = 32,     # bây giờ spawn sẽ cho phép dùng CUDA safe
        pin_memory   = True,
    )

    print(f"Số mẫu trong test set: {len(test_loader.dataset)}")
    print(f"Số batch trong test loader: {len(test_loader)}")

    # --- Chạy đánh giá ---
    metrics = trainer.test_epoch_ver4(test_loader, epoch=0)
    print(f"Individual metrics: {metrics}")


Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:17<00:00,  1.09s/it]

▶ Full Text→Image:     73.6000%
▶ Image→Full Text:     76.2000%
▶ 1th Sentence→Image:  73.5000%
▶ 2th Sentence→Image:  0.2000%
▶ 3th Sentence→Image:  0.1000%
▶ 4th Sentence→Image:  0.1000%
Individual metrics: [0.736, 0.762, 0.735, 0.002, 0.001, 0.001]


: 

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import json
import pandas as pd
from PIL import Image
import clip
import torch
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace
from tqdm import tqdm
from train import CLIP_Clean_Train
from dci import JsonDCIDataset
from docci import DocciDataset
# from sharegpt4v_test import JsonSHAREGPT4VTESTDataset

# 1) Thiết lập multiprocessing start method
# import torch.multiprocessing as mp
# try:
#     mp.set_start_method('spawn', force=True)
# except RuntimeError:
#     pass

class Flickr30kPairDataset(Dataset):
    """
    Mỗi caption thành một sample riêng, và chỉ lấy split bạn muốn (train/val/test).
    Trả về: image_tensor, caption_long, caption_short, img_path, split
    """
    def __init__(
        self,
        csv_path: str,
        root_dir: str,
        clip_model: str = "ViT-B/16",
        split: str = None,
        max_items: int = None,
    ):
        super().__init__()
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        if max_items is not None:
            df = df.iloc[:max_items]
        df = df.reset_index(drop=True)

        samples = []
        for _, row in df.iterrows():
            img_path = os.path.join(root_dir, "flickr30k-images", row["filename"])
            caps = json.loads(row["raw"])
            # for cap in caps:
            samples.append({
                "img_path": img_path,
                "caption": caps[0].strip(),
                "split": row["split"],
                "img_id": row["img_id"],
            })
        self.samples = samples

        _, self.preprocess = clip.load(clip_model, device="cpu")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["img_path"]).convert("RGB")
        image_tensor = self.preprocess(image)
        caption_long = item["caption"]
        caption_short = item["caption"]
        return image_tensor, caption_long, caption_short, item["img_path"], item["split"]
    


# csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
# root = "/home/ubuntu/shared/hieu.tq/flickr30k"
# ds = Flickr30kPairDataset(csv_file, root, max_items=100)
# loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=True)

# batch = next(iter(loader))

# print("Batch images:", batch["image"].shape)    # [16,3,224,224]
# print("Batch captions:", batch["caption"][:5]) # 5 câu đầu
# print("Paths:", batch["img_path"][:3])

if __name__ == "__main__":
    # --- Inline args và load model ---
    args = SimpleNamespace(
        base_model='ViT-B/16',
        download_root=None,
        log_scale=4.6052,
        batch_size=64,
        epochs=1,
        lr=1e-5,
        refine_lr_image=2e-6,
        refine_lr_text=2e-4,
        weight_decay=0.01,
        warmup_length=200,
        exp_name='eval_run',
        ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-20--18_41_54_-docci-no.pt"
    )

    trainer = CLIP_Clean_Train(args)
    state = torch.load(args.ckpt_path, map_location='cpu')
    trainer.model.load_state_dict(state, strict=False)

    # --- Khởi tạo dataset + loader ---
    csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
    root_dir = "/home/ubuntu/shared/hieu.tq/flickr30k"
    test_ds = Flickr30kPairDataset(
        csv_path=csv_file,
        root_dir=root_dir,
        clip_model="ViT-B/16",
        split="test",
        max_items=None
    )
    # test_ds = DocciDataset(split='test', max_items=None)

    test_loader = DataLoader(
        test_ds,
        batch_size   = args.batch_size,
        shuffle      = False,
        num_workers  = 32,     # bây giờ spawn sẽ cho phép dùng CUDA safe
        pin_memory   = True,
    )

    print(f"Số mẫu trong test set: {len(test_loader.dataset)}")
    print(f"Số batch trong test loader: {len(test_loader)}")

    # --- Chạy đánh giá ---
    metrics = trainer.test_epoch_ver4(test_loader, epoch=0)
    print(f"Individual metrics: {metrics}")


Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:27<00:00,  1.75s/it]

▶ Full Text→Image:     72.4000%
▶ Image→Full Text:     73.6000%
▶ 1th Sentence→Image:  72.1000%
▶ 2th Sentence→Image:  0.1000%
▶ 3th Sentence→Image:  0.1000%
▶ 4th Sentence→Image:  0.1000%
Individual metrics: [0.724, 0.736, 0.721, 0.001, 0.001, 0.001]


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import json
import pandas as pd
from PIL import Image
import clip
import torch
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace
from tqdm import tqdm
from train import CLIP_Clean_Train
from dci import JsonDCIDataset
from docci import DocciDataset
# from sharegpt4v_test import JsonSHAREGPT4VTESTDataset

# 1) Thiết lập multiprocessing start method
# import torch.multiprocessing as mp
# try:
#     mp.set_start_method('spawn', force=True)
# except RuntimeError:
#     pass

class Flickr30kPairDataset(Dataset):
    """
    Mỗi caption thành một sample riêng, và chỉ lấy split bạn muốn (train/val/test).
    Trả về: image_tensor, caption_long, caption_short, img_path, split
    """
    def __init__(
        self,
        csv_path: str,
        root_dir: str,
        clip_model: str = "ViT-B/16",
        split: str = None,
        max_items: int = None,
    ):
        super().__init__()
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        if max_items is not None:
            df = df.iloc[:max_items]
        df = df.reset_index(drop=True)

        samples = []
        for _, row in df.iterrows():
            img_path = os.path.join(root_dir, "flickr30k-images", row["filename"])
            caps = json.loads(row["raw"])
            # for cap in caps:
            samples.append({
                "img_path": img_path,
                "caption": caps[0].strip(),
                "split": row["split"],
                "img_id": row["img_id"],
            })
        self.samples = samples

        _, self.preprocess = clip.load(clip_model, device="cpu")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["img_path"]).convert("RGB")
        image_tensor = self.preprocess(image)
        caption_long = item["caption"]
        caption_short = item["caption"]
        return image_tensor, caption_long, caption_short, item["img_path"], item["split"]
    


# csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
# root = "/home/ubuntu/shared/hieu.tq/flickr30k"
# ds = Flickr30kPairDataset(csv_file, root, max_items=100)
# loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=True)

# batch = next(iter(loader))

# print("Batch images:", batch["image"].shape)    # [16,3,224,224]
# print("Batch captions:", batch["caption"][:5]) # 5 câu đầu
# print("Paths:", batch["img_path"][:3])

if __name__ == "__main__":
    # --- Inline args và load model ---
    args = SimpleNamespace(
        base_model='ViT-B/16',
        download_root=None,
        log_scale=4.6052,
        batch_size=64,
        epochs=1,
        lr=1e-5,
        refine_lr_image=2e-6,
        refine_lr_text=2e-4,
        weight_decay=0.01,
        warmup_length=200,
        exp_name='eval_run',
        ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-23--11_35_11_-dci-no-5.pt"
    )

    trainer = CLIP_Clean_Train(args)
    state = torch.load(args.ckpt_path, map_location='cpu')
    trainer.model.load_state_dict(state, strict=False)

    # --- Khởi tạo dataset + loader ---
    csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
    root_dir = "/home/ubuntu/shared/hieu.tq/flickr30k"
    test_ds = Flickr30kPairDataset(
        csv_path=csv_file,
        root_dir=root_dir,
        clip_model="ViT-B/16",
        split="test",
        max_items=None
    )
    # test_ds = DocciDataset(split='test', max_items=None)

    test_loader = DataLoader(
        test_ds,
        batch_size   = args.batch_size,
        shuffle      = False,
        num_workers  = 32,     # bây giờ spawn sẽ cho phép dùng CUDA safe
        pin_memory   = True,
    )

    print(f"Số mẫu trong test set: {len(test_loader.dataset)}")
    print(f"Số batch trong test loader: {len(test_loader)}")

    # --- Chạy đánh giá ---
    metrics = trainer.test_epoch_ver4(test_loader, epoch=0)
    print(f"Individual metrics: {metrics}")


/home/ubuntu/miniconda3/envs/kdpl/lib/python3.8/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Số mẫu trong test set: 1000
Số batch trong test loader: 16


Extracting features: 100%|██████████| 16/16 [00:17<00:00,  1.10s/it]

▶ Full Text→Image:     71.5000%
▶ Image→Full Text:     72.2000%
▶ 1th Sentence→Image:  71.3000%
▶ 2th Sentence→Image:  0.2000%
▶ 3th Sentence→Image:  0.1000%
▶ 4th Sentence→Image:  0.1000%
Individual metrics: [0.715, 0.722, 0.713, 0.002, 0.001, 0.001]


In [ ]:
import os
import json
import pandas as pd
from PIL import Image
import clip
import torch
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace
from tqdm import tqdm
from train import CLIP_Clean_Train
from dci import JsonDCIDataset
from docci import DocciDataset
# from sharegpt4v_test import JsonSHAREGPT4VTESTDataset

# 1) Thiết lập multiprocessing start method
# import torch.multiprocessing as mp
# try:
#     mp.set_start_method('spawn', force=True)
# except RuntimeError:
#     pass

class Flickr30kPairDataset(Dataset):
    """
    Mỗi caption thành một sample riêng, và chỉ lấy split bạn muốn (train/val/test).
    Trả về: image_tensor, caption_long, caption_short, img_path, split
    """
    def __init__(
        self,
        csv_path: str,
        root_dir: str,
        clip_model: str = "ViT-B/16",
        split: str = None,
        max_items: int = None,
    ):
        super().__init__()
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        if max_items is not None:
            df = df.iloc[:max_items]
        df = df.reset_index(drop=True)

        samples = []
        for _, row in df.iterrows():
            img_path = os.path.join(root_dir, "flickr30k-images", row["filename"])
            caps = json.loads(row["raw"])
            # for cap in caps:
            samples.append({
                "img_path": img_path,
                "caption": caps[0].strip(),
                "split": row["split"],
                "img_id": row["img_id"],
            })
        self.samples = samples

        _, self.preprocess = clip.load(clip_model, device="cpu")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["img_path"]).convert("RGB")
        image_tensor = self.preprocess(image)
        caption_long = item["caption"]
        caption_short = item["caption"]
        return image_tensor, caption_long, caption_short, item["img_path"], item["split"]
    


# csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
# root = "/home/ubuntu/shared/hieu.tq/flickr30k"
# ds = Flickr30kPairDataset(csv_file, root, max_items=100)
# loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=True)

# batch = next(iter(loader))

# print("Batch images:", batch["image"].shape)    # [16,3,224,224]
# print("Batch captions:", batch["caption"][:5]) # 5 câu đầu
# print("Paths:", batch["img_path"][:3])

if __name__ == "__main__":
    # --- Inline args và load model ---
    args = SimpleNamespace(
        base_model='ViT-B/16',
        download_root=None,
        log_scale=4.6052,
        batch_size=64,
        epochs=1,
        lr=1e-5,
        refine_lr_image=2e-6,
        refine_lr_text=2e-4,
        weight_decay=0.01,
        warmup_length=200,
        exp_name='eval_run',
        ckpt_path="/home/ubuntu/hieu.tq/Git/KDPL_test/KDPL/src/LongCLIPMul_docci/train/longclip/lr=1e-06_wd=0.01_wl=200_logs=4.6052_64xb/ckpt/Propose-b16-longclip-06-20--18_41_54_-docci-no.pt"
    )

    trainer = CLIP_Clean_Train(args)
    state = torch.load(args.ckpt_path, map_location='cpu')
    trainer.model.load_state_dict(state, strict=False)

    # --- Khởi tạo dataset + loader ---
    csv_file = "/home/ubuntu/shared/hieu.tq/flickr30k/flickr_annotations_30k.csv"
    root_dir = "/home/ubuntu/shared/hieu.tq/flickr30k"
    test_ds = Flickr30kPairDataset(
        csv_path=csv_file,
        root_dir=root_dir,
        clip_model="ViT-B/16",
        split="test",
        max_items=None
    )
    # test_ds = DocciDataset(split='test', max_items=None)

    test_loader = DataLoader(
        test_ds,
        batch_size   = args.batch_size,
        shuffle      = False,
        num_workers  = 32,     # bây giờ spawn sẽ cho phép dùng CUDA safe
        pin_memory   = True,
    )

    print(f"Số mẫu trong test set: {len(test_loader.dataset)}")
    print(f"Số batch trong test loader: {len(test_loader)}")

    # --- Chạy đánh giá ---
    metrics = trainer.test_epoch_ver4(test_loader, epoch=0)
    print(f"Individual metrics: {metrics}")


# new 25/06